In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense
from keras.optimizers import Adam

import warnings
warnings.filterwarnings('ignore')

Using TensorFlow backend.


In [2]:
train_df = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
test_df = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')

In [3]:
train_df.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
train_df.shape,test_df.shape

((42000, 785), (28000, 784))

In [5]:
X = train_df.iloc[:,1:785].values
y = train_df.iloc[:,0].values

final_test = test_df.values

X = X/255
final_test = final_test/255

y = to_categorical(y)

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3)

In [6]:
network = Sequential()
network.add(Dense(512,input_dim=X_train.shape[1],activation='relu'))
network.add(Dense(128, activation = 'relu'))
network.add(Dense(10,activation='softmax'))

In [7]:
#complie the network
#To get network ready to fit to the training data, you have to first compile it. This involves
#specifying the optimizer (a choice of strategies to apply to solve for the network parameters),
#the loss function to minimize (categorical cross-entropy in this case as is common for multi-class 
#classification problems), and a choice of metrics to track in the iterative process.

network.compile(optimizer=Adam(lr=1e-4),loss='categorical_crossentropy',metrics=['accuracy'])
#network.compile(loss="categorical_crossentropy",
#              optimizer=optimizers.Adadelta(lr=1.0, rho=0.95, epsilon=1e-08, decay=0.0),
#              metrics=['accuracy'])

In [8]:
#Fit the model with training data
#As optional keyword arguments, specify epochs=5 (the number of sweeps through the data to make) and 
#batch_size=128 (the number of data points to use in each sweep through the data). This is in principle
#the same as the iterations of stochastic gradient descent (with a batch size of 1) 
#made in the perceptron algorithm.

history = network.fit(X_train,y_train,epochs=20,batch_size=128,verbose=0)


In [9]:
scores = network.evaluate(X_test,y_test)
   
print("Loss=" + str(scores[0]))
print("Accuracy=" + str(scores[1]))

12600/12600 [==============================] - 1s 48us/step
Loss=0.09515138968707076
Accuracy=0.971666693687439


In [10]:
#predicted_classes = network.predict_classes(final_test)

y_pred = network.predict(final_test)
predicted_classes = np.argmax(y_pred,axis=1)

submissions=pd.DataFrame({"ImageId": list(range(1,len(predicted_classes)+1)),
                         "Label": predicted_classes})
submissions.to_csv("my_digit.csv", index=False, header=True)

In [11]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


train_img   = train_df.iloc[:,1:785].values
train_label = train_df.iloc[:,0].values

test_img    = test_df.values

scaler  = StandardScaler()

scaler.fit(train_img)

train_df = scaler.transform(train_img)
test_df = scaler.transform(test_img)

pca = PCA(0.95)

pca.fit(train_img)

n = pca.components_
print(len(n))

train_img = pca.transform(train_img)
test_img = pca.transform(test_img)
final_test = pca.transform(final_test)

154


In [12]:
log_Reg = LogisticRegression(solver='lbfgs')

log_Reg.fit(train_img,train_label)


LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
                   intercept_scaling=1, l1_ratio=None, max_iter=100,
                   multi_class='warn', n_jobs=None, penalty='l2',
                   random_state=None, solver='lbfgs', tol=0.0001, verbose=0,
                   warm_start=False)

In [13]:
#predicted_classes = network.predict_classes(final_test)

y_pred = log_Reg.predict(test_img)
#predicted_classes = np.argmax(y_pred,axis=1)

submissions=pd.DataFrame({"ImageId": list(range(1,len(y_pred)+1)),
                         "Label": y_pred})
submissions.to_csv("my_digit_pca.csv", index=False, header=True)